# Caso práctico: Limpieza de una grabación contaminada

**Asignatura:** Sistemas Multimedia  
**Caso:** Filtrado de una grabación de voz degradada por interferencias eléctricas y ruido de alta frecuencia  

---

## Descripción del problema

Supongamos que recibimos una grabación de voz (o cualquier señal útil de banda media)
que ha sido captada con un equipo mal apantallado. El archivo resultante presenta dos
tipos de contaminación simultánea:

1. **Zumbido eléctrico (hum):** interferencia de la red eléctrica europea a **50 Hz**
   y su primer armónico a **100 Hz**. Es el ruido más común en grabaciones de interior.
2. **Ruido de alta frecuencia:** chirridos a **8 000 Hz** y **12 000 Hz** producidos,
   por ejemplo, por la electrónica del convertidor A/D o por interferencias de RF.

La señal útil que queremos recuperar contiene componentes entre **200 Hz y 4 000 Hz**
(rango típico de la voz humana).

**Objetivo:** aplicar una cadena de dos filtros Butterworth para aislar la señal útil,
medir la mejora con métricas de calidad (SNR, THD, rango dinámico) y visualizar
cada paso con espectros y espectrogramas.

---

## Índice

1. Generación de la señal sintética contaminada  
2. Análisis inicial: onda temporal, espectro y espectrograma  
3. Paso 1 — Filtro paso alto (eliminar hum eléctrico)  
4. Paso 2 — Filtro paso bajo (eliminar ruido de alta frecuencia)  
5. Comparativa de métricas antes y después  
6. Conclusiones  


## 0. Importaciones y configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.io import wavfile
from scipy.signal import butter, sosfilt
from scipy.fft import rfft, rfftfreq
import soundfile as sf
import warnings
warnings.filterwarnings('ignore')

# Parámetros globales de figura
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor':   '#1e1e2e',
    'axes.edgecolor':   '#555',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#ccc',
    'ytick.color':      '#ccc',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

SR  = 44100   # sample rate
DUR = 4       # segundos
t   = np.linspace(0, DUR, SR * DUR, endpoint=False)

print(f"Configuración: SR={SR} Hz, duración={DUR} s, muestras={len(t):,}")


## 1. Generación de la señal sintética contaminada

Construimos la señal sumando tres capas:

| Componente | Frecuencia | Amplitud | Origen |
|---|---|---|---|
| Señal útil (voz simulada) | 300, 800, 2 000 Hz | 0.30 cada una | Contenido que queremos conservar |
| Hum eléctrico | 50, 100 Hz | 0.80, 0.45 | Red eléctrica 50 Hz + 1er armónico |
| Ruido HF | 8 000, 12 000 Hz | 0.35, 0.25 | Interferencia electrónica |

La señal útil tiene amplitud total ≈ 0.9, pero el hum la supera en la suma final,
simulando una grabación donde el ruido enmascara la voz.


In [ ]:
# ── Señal útil: tres tonos en rango de voz ──
voz = (0.30 * np.sin(2 * np.pi * 300  * t) +
       0.30 * np.sin(2 * np.pi * 800  * t) +
       0.30 * np.sin(2 * np.pi * 2000 * t))

# ── Interferencia de red eléctrica (hum 50 Hz + armónico) ──
hum = (0.80 * np.sin(2 * np.pi * 50  * t) +
       0.45 * np.sin(2 * np.pi * 100 * t))

# ── Ruido de alta frecuencia ──
hf_noise = (0.35 * np.sin(2 * np.pi * 8000  * t) +
            0.25 * np.sin(2 * np.pi * 12000 * t))

# ── Señal contaminada final ──
señal_contaminada = voz + hum + hf_noise

# Normalizar a [-1, 1] para evitar clipping
señal_contaminada /= np.max(np.abs(señal_contaminada))

# Guardar como WAV para poder cargarlo en la app también
sf.write('señal_contaminada.wav', señal_contaminada.astype(np.float32), SR)

print("Potencia por componente:")
print(f"  Voz útil : {np.mean(voz**2):.5f}")
print(f"  Hum 50Hz : {np.mean(hum**2):.5f}")
print(f"  Ruido HF : {np.mean(hf_noise**2):.5f}")
print(f"  Total    : {np.mean(señal_contaminada**2):.5f}")
print("\nArchivo guardado: señal_contaminada.wav")


## 2. Funciones auxiliares de análisis

In [ ]:
def espectro(signal, sr, ax, color='#4e8ef7', label=''):
    """Dibuja el espectro de potencia en dB sobre un eje dado."""
    N  = len(signal)
    yf = np.abs(rfft(signal)) * 2 / N
    xf = rfftfreq(N, 1 / sr)
    ax.plot(xf, 20 * np.log10(yf + 1e-10), color=color, linewidth=0.9, label=label)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Magnitud (dB)")
    ax.grid(True)
    ax.set_xlim(0, sr // 2)


def espectrograma(signal, sr, ax, n_fft=1024, hop=256):
    """Dibuja el espectrograma STFT sobre un eje dado."""
    window = np.hanning(n_fft)
    frames = [np.abs(rfft(signal[i:i+n_fft] * window))
              for i in range(0, len(signal) - n_fft, hop)]
    S_db  = 20 * np.log10(np.array(frames).T + 1e-10)
    freqs = rfftfreq(n_fft, 1 / sr)
    times = np.arange(S_db.shape[1]) * hop / sr
    im = ax.pcolormesh(times, freqs, S_db, cmap='magma', shading='auto',
                       vmin=-80, vmax=0)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Hz)")
    return im


def butter_filter(signal, cutoff, btype, sr, order=5):
    """Aplica un filtro Butterworth y devuelve la señal filtrada."""
    nyq = sr / 2.0
    wn  = [c / nyq for c in cutoff] if isinstance(cutoff, (list, tuple)) else cutoff / nyq
    sos = butter(order, wn, btype=btype, output='sos')
    return sosfilt(sos, signal).astype(np.float32)


def calcular_metricas(signal, sr):
    """Calcula SNR (mediana espectral), THD y rango dinámico."""
    N           = len(signal)
    yf_pow      = np.abs(rfft(signal)) ** 2 * 2 / N
    yf_amp      = np.sqrt(yf_pow)
    xf          = rfftfreq(N, 1 / sr)

    # SNR por suelo espectral
    sig_pow     = float(np.mean(signal ** 2))
    noise_floor = float(np.median(yf_pow))
    noise_pow   = noise_floor * (N // 2) / N
    snr = 10 * np.log10(sig_pow / (noise_pow + 1e-10))

    # THD
    f0  = float(xf[np.argmax(yf_amp)])
    def amp(f): return float(yf_amp[np.argmin(np.abs(xf - f))])
    v1  = amp(f0)
    thd = (np.sqrt(sum(amp(f0*k)**2 for k in range(2, 6))) / (v1 + 1e-10)) * 100

    # Rango dinámico
    pico     = np.max(np.abs(signal))
    non_zero = np.abs(signal[np.abs(signal) > 1e-10])
    minimo   = np.min(non_zero) if len(non_zero) else 1e-10
    dr       = 20 * np.log10(pico + 1e-10) - 20 * np.log10(minimo + 1e-10)

    return {"SNR (dB)": round(snr, 2),
            "THD (%)":  round(thd, 4),
            "DR (dB)":  round(dr,  2)}


print("Funciones definidas: espectro(), espectrograma(), butter_filter(), calcular_metricas()")


## 3. Análisis inicial de la señal contaminada

Antes de aplicar ningún filtro, visualizamos la señal en tres representaciones:

- **Onda temporal:** muestra la forma de onda global. El hum de 50 Hz produce
  una oscilación lenta que domina visualmente.
- **Espectro de frecuencia:** permite identificar exactamente los picos de cada
  componente. Es la herramienta clave para decidir qué frecuencias de corte usar.
- **Espectrograma:** muestra cómo varía el contenido frecuencial a lo largo del tiempo.
  Al ser tonos puros sintéticos, veremos líneas horizontales constantes.


In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 1, hspace=0.5)

# ── Onda temporal (decimada para velocidad) ──
ax0 = fig.add_subplot(gs[0])
step = max(1, len(t) // 8000)
ax0.plot(t[::step], señal_contaminada[::step], color='#4e8ef7', linewidth=0.7)
ax0.set_title("Señal contaminada — Onda temporal", pad=8)
ax0.set_xlabel("Tiempo (s)")
ax0.set_ylabel("Amplitud")
ax0.grid(True)
ax0.axhline(0, color='#666', linewidth=0.5)

# ── Espectro ──
ax1 = fig.add_subplot(gs[1])
espectro(señal_contaminada, SR, ax1, color='#e05c5c')
ax1.set_title("Señal contaminada — Espectro de frecuencia", pad=8)
ax1.set_xlim(0, 15000)

# Marcar los picos identificados
for freq, etiq, col in [
    (50,    "50 Hz\n(hum)",     '#f59e0b'),
    (100,   "100 Hz\n(arm.)",   '#f59e0b'),
    (300,   "300 Hz\n(voz)",    '#4ade80'),
    (800,   "800 Hz\n(voz)",    '#4ade80'),
    (2000,  "2kHz\n(voz)",      '#4ade80'),
    (8000,  "8kHz\n(HF)",       '#f87171'),
    (12000, "12kHz\n(HF)",      '#f87171'),
]:
    ax1.axvline(freq, color=col, linewidth=1, linestyle='--', alpha=0.7)
    ax1.text(freq + 80, -5, etiq, color=col, fontsize=7, va='top')

# Leyenda de colores
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#f59e0b', linestyle='--', label='Hum eléctrico (eliminar)'),
    Line2D([0], [0], color='#4ade80', linestyle='--', label='Señal útil (conservar)'),
    Line2D([0], [0], color='#f87171', linestyle='--', label='Ruido HF (eliminar)'),
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=8,
           facecolor='#2a2a3e', edgecolor='#555')

# ── Espectrograma ──
ax2 = fig.add_subplot(gs[2])
im = espectrograma(señal_contaminada, SR, ax2)
ax2.set_title("Señal contaminada — Espectrograma", pad=8)
ax2.set_ylim(0, 14000)
fig.colorbar(im, ax=ax2, label='dB', pad=0.01)

plt.suptitle("Análisis inicial — Señal contaminada", fontsize=13, y=1.01, color='#eee')
plt.savefig("analisis_inicial.png", dpi=120, bbox_inches='tight',
            facecolor='#1e1e2e')
plt.show()
print("Figura guardada: analisis_inicial.png")


### Métricas de la señal contaminada

Calculamos las métricas base que usaremos como referencia para medir la mejora
después de cada filtro.


In [ ]:
metricas_original = calcular_metricas(señal_contaminada, SR)
print("Métricas — Señal CONTAMINADA (referencia)")
print("-" * 40)
for k, v in metricas_original.items():
    print(f"  {k:<12}: {v}")


## 4. Paso 1 — Filtro paso alto (eliminación del hum eléctrico)

### Justificación del filtro

El hum eléctrico ocupa las frecuencias **50 Hz y 100 Hz**. Nuestra señal útil
empieza en **300 Hz**, por lo que tenemos margen suficiente para colocar la
frecuencia de corte en **150 Hz** sin afectar el contenido de interés.

| Parámetro | Valor | Razonamiento |
|---|---|---|
| Tipo | Paso alto | Queremos eliminar lo que está *por debajo* de un umbral |
| Fc | 150 Hz | Margen cómodo entre 100 Hz (último armónico) y 300 Hz (primera voz) |
| Orden | 6 | Pendiente suficientemente agresiva para atenuar 50 y 100 Hz sin distorsionar 300 Hz |

**Respuesta en frecuencia esperada:**
- A 50 Hz (2 octavas por debajo de 150 Hz) → atenuación ≈ −36 dB (orden 6)
- A 100 Hz (≈ 0.7 octavas por debajo) → atenuación ≈ −18 dB
- A 300 Hz (1 octava por encima) → prácticamente sin atenuación


In [ ]:
FC_HIGHPASS = 150   # Hz
ORDEN_HP    = 6

señal_paso1 = butter_filter(señal_contaminada, FC_HIGHPASS, 'high', SR, ORDEN_HP)

print(f"Filtro paso alto aplicado: Fc = {FC_HIGHPASS} Hz, orden = {ORDEN_HP}")
print(f"Atenuación teórica a 50 Hz  : {-20*ORDEN_HP*np.log10(FC_HIGHPASS/50):.1f} dB")
print(f"Atenuación teórica a 100 Hz : {-20*ORDEN_HP*np.log10(FC_HIGHPASS/100):.1f} dB")
print(f"Atenuación teórica a 300 Hz : ~0 dB (dentro de la banda de paso)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Espectro comparativo
espectro(señal_contaminada, SR, axes[0], color='#e05c5c', label='Contaminada')
espectro(señal_paso1,       SR, axes[0], color='#4e8ef7', label='Tras paso alto')
axes[0].set_title("Espectro — Antes vs. después del paso alto")
axes[0].set_xlim(0, 500)   # Zoom en zona baja para ver el efecto del hum
axes[0].legend(facecolor='#2a2a3e', edgecolor='#555', fontsize=9)
axes[0].axvline(FC_HIGHPASS, color='#f59e0b', linestyle=':', linewidth=1.5,
                label=f'Fc = {FC_HIGHPASS} Hz')

# Espectro completo
espectro(señal_contaminada, SR, axes[1], color='#e05c5c', label='Contaminada')
espectro(señal_paso1,       SR, axes[1], color='#4e8ef7', label='Tras paso alto')
axes[1].set_title("Espectro completo (0 – 22 kHz)")
axes[1].set_xlim(0, SR // 2)
axes[1].legend(facecolor='#2a2a3e', edgecolor='#555', fontsize=9)

plt.suptitle("Resultado del filtro paso alto (Fc = 150 Hz, orden 6)",
             fontsize=12, color='#eee')
plt.tight_layout()
plt.savefig("paso1_highpass.png", dpi=120, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print("Los picos de 50 y 100 Hz deben haber desaparecido en el panel izquierdo.")


In [ ]:
metricas_paso1 = calcular_metricas(señal_paso1, SR)
print("Métricas — Tras filtro PASO ALTO")
print("-" * 40)
for k, v in metricas_paso1.items():
    delta = v - metricas_original[k]
    signo = '+' if delta >= 0 else ''
    print(f"  {k:<12}: {v}  (Δ {signo}{delta:.2f})")


## 5. Paso 2 — Filtro paso bajo (eliminación del ruido de alta frecuencia)

### Justificación del filtro

Tras el paso alto hemos eliminado el hum. Ahora eliminamos el ruido de HF.
La señal útil llega hasta **2 000 Hz**; el ruido empieza en **8 000 Hz**.
Colocamos la frecuencia de corte en **4 000 Hz** para conservar todo el
rango de la voz con margen amplio.

| Parámetro | Valor | Razonamiento |
|---|---|---|
| Tipo | Paso bajo | Queremos eliminar lo que está *por encima* de un umbral |
| Fc | 4 000 Hz | Cubre el rango de voz (hasta 2 kHz) con 1 octava de margen |
| Orden | 5 | Suficiente para atenuar fuertemente 8 kHz y 12 kHz |

**Respuesta en frecuencia esperada:**
- A 4 000 Hz (frecuencia de corte) → −3 dB (por definición)
- A 8 000 Hz (1 octava por encima) → atenuación ≈ −30 dB (orden 5)
- A 12 000 Hz → atenuación > −40 dB


In [ ]:
FC_LOWPASS = 4000   # Hz
ORDEN_LP   = 5

señal_final = butter_filter(señal_paso1, FC_LOWPASS, 'low', SR, ORDEN_LP)

print(f"Filtro paso bajo aplicado: Fc = {FC_LOWPASS} Hz, orden = {ORDEN_LP}")
print(f"Atenuación teórica a 8000 Hz  : {-20*ORDEN_LP*np.log10(8000/FC_LOWPASS):.1f} dB")
print(f"Atenuación teórica a 12000 Hz : {-20*ORDEN_LP*np.log10(12000/FC_LOWPASS):.1f} dB")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Zoom zona alta para ver el efecto
espectro(señal_paso1,  SR, axes[0], color='#4e8ef7', label='Tras paso alto')
espectro(señal_final,  SR, axes[0], color='#a78bfa', label='Tras paso bajo')
axes[0].set_title("Espectro — Zoom zona HF (4 – 15 kHz)")
axes[0].set_xlim(4000, 15000)
axes[0].axvline(FC_LOWPASS, color='#f59e0b', linestyle=':', linewidth=1.5,
                label=f'Fc = {FC_LOWPASS} Hz')
axes[0].legend(facecolor='#2a2a3e', edgecolor='#555', fontsize=9)

# Espectro completo de los tres estados
espectro(señal_contaminada, SR, axes[1], color='#e05c5c', label='Contaminada')
espectro(señal_paso1,       SR, axes[1], color='#4e8ef7', label='Paso alto')
espectro(señal_final,       SR, axes[1], color='#a78bfa', label='Paso alto + bajo')
axes[1].set_title("Espectro completo — Evolución de los tres estados")
axes[1].set_xlim(0, SR // 2)
axes[1].legend(facecolor='#2a2a3e', edgecolor='#555', fontsize=9)

plt.suptitle("Resultado del filtro paso bajo (Fc = 4 000 Hz, orden 5)",
             fontsize=12, color='#eee')
plt.tight_layout()
plt.savefig("paso2_lowpass.png", dpi=120, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print("Los picos de 8 000 y 12 000 Hz deben haber desaparecido.")


In [ ]:
metricas_final = calcular_metricas(señal_final, SR)
print("Métricas — Señal FINAL (paso alto + paso bajo)")
print("-" * 40)
for k, v in metricas_final.items():
    delta = v - metricas_original[k]
    signo = '+' if delta >= 0 else ''
    print(f"  {k:<12}: {v}  (Δ {signo}{delta:.2f})")


## 6. Comparativa visual completa

Superponemos los espectros y espectrogramas de los tres estados para
ver la evolución completa del proceso de limpieza.


In [ ]:
fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(3, 2, hspace=0.55, wspace=0.35)

estados = [
    (señal_contaminada, '#e05c5c', 'Contaminada'),
    (señal_paso1,       '#4e8ef7', 'Tras paso alto (Fc=150 Hz)'),
    (señal_final,       '#a78bfa', 'Tras paso alto + bajo (Fc=4 kHz)'),
]

for fila, (sig, color, titulo) in enumerate(estados):
    # Espectro
    ax_esp = fig.add_subplot(gs[fila, 0])
    espectro(sig, SR, ax_esp, color=color)
    ax_esp.set_title(f"Espectro — {titulo}", fontsize=9, pad=5)
    ax_esp.set_xlim(0, 14000)
    ax_esp.set_ylim(-90, 5)

    # Espectrograma
    ax_sgm = fig.add_subplot(gs[fila, 1])
    im = espectrograma(sig, SR, ax_sgm)
    ax_sgm.set_title(f"Espectrograma — {titulo}", fontsize=9, pad=5)
    ax_sgm.set_ylim(0, 14000)
    fig.colorbar(im, ax=ax_sgm, label='dB', pad=0.01, shrink=0.85)

plt.suptitle("Evolución del proceso de filtrado", fontsize=13, color='#eee', y=1.01)
plt.savefig("comparativa_completa.png", dpi=120, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print("Figura guardada: comparativa_completa.png")


## 7. Tabla comparativa de métricas

Resumimos la evolución de SNR, THD y rango dinámico en los tres estados.


In [ ]:
import pandas as pd

tabla = pd.DataFrame({
    'Estado'     : ['Contaminada', 'Paso alto (150 Hz)', 'Paso alto + bajo (4 kHz)'],
    'SNR (dB)'   : [metricas_original['SNR (dB)'],
                    metricas_paso1['SNR (dB)'],
                    metricas_final['SNR (dB)']],
    'THD (%)'    : [metricas_original['THD (%)'],
                    metricas_paso1['THD (%)'],
                    metricas_final['THD (%)']],
    'DR (dB)'    : [metricas_original['DR (dB)'],
                    metricas_paso1['DR (dB)'],
                    metricas_final['DR (dB)']],
})

# Añadir columnas de mejora respecto al original
tabla['ΔSNR']  = (tabla['SNR (dB)'] - tabla.loc[0, 'SNR (dB)']).round(2)
tabla['ΔTHD']  = (tabla['THD (%)']  - tabla.loc[0, 'THD (%)']).round(4)
tabla['ΔDR']   = (tabla['DR (dB)']  - tabla.loc[0, 'DR (dB)']).round(2)

print(tabla.to_string(index=False))


In [ ]:
# Gráfico de barras comparativo
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
etiquetas = ['Contaminada', 'Paso\nalto', 'Paso alto\n+ bajo']
colores   = ['#e05c5c', '#4e8ef7', '#a78bfa']

metricas_list = [metricas_original, metricas_paso1, metricas_final]

for ax, metrica, titulo, unidad in zip(
    axes,
    ['SNR (dB)', 'THD (%)', 'DR (dB)'],
    ['SNR — Relación señal/ruido',
     'THD — Distorsión armónica total',
     'DR — Rango dinámico'],
    ['dB', '%', 'dB']
):
    valores = [m[metrica] for m in metricas_list]
    bars = ax.bar(etiquetas, valores, color=colores, edgecolor='#333', width=0.5)
    ax.set_title(titulo, fontsize=9, pad=6)
    ax.set_ylabel(unidad)
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', va='bottom', fontsize=8, color='#eee')

plt.suptitle("Comparativa de métricas de calidad", fontsize=12, color='#eee')
plt.tight_layout()
plt.savefig("metricas_comparativa.png", dpi=120, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()


## 8. Exportación del resultado

In [ ]:
sf.write('señal_limpia.wav', señal_final.astype(np.float32), SR)
print("Archivos generados:")
print("  señal_contaminada.wav  — señal de entrada con hum y ruido HF")
print("  señal_limpia.wav       — señal tras la cadena de filtros")
print()
print("Ambos archivos son cargables directamente en la aplicación para")
print("comparar su onda, espectro y espectrograma de forma interactiva.")


## 9. Conclusiones

### Resultados del proceso de filtrado

La cadena de dos filtros Butterworth aplicada ha conseguido:

1. **Eliminación del hum eléctrico (50/100 Hz):** el filtro paso alto con Fc = 150 Hz
   y orden 6 atenúa el componente de 50 Hz más de 30 dB y el de 100 Hz más de 18 dB,
   haciéndolos inaudibles en la señal resultante.

2. **Eliminación del ruido de alta frecuencia (8/12 kHz):** el filtro paso bajo con
   Fc = 4 000 Hz y orden 5 atenúa ambos componentes más de 30 dB, eliminándolos
   por completo del espectro visible.

3. **Conservación de la señal útil:** los tonos a 300, 800 y 2 000 Hz se mantienen
   intactos, ya que ambas frecuencias de corte se eligieron con margen suficiente.

### Métricas

| Métrica | Interpretación del cambio |
|---|---|
| **SNR** | Aumenta porque el ruido (hum + HF) se elimina, mejorando la relación señal/fondo espectral |
| **THD** | Cambia porque se eliminan armónicos que antes contribuían al cálculo |
| **DR** | Aumenta porque al eliminar el hum dominante, el mínimo de la señal sube y el rango se amplía |

### Consideraciones de diseño

- **Elección de Fc:** siempre debe haber al menos media octava de margen entre
  la última frecuencia de ruido y la primera frecuencia útil.
- **Orden del filtro:** órdenes mayores implican pendientes más agresivas pero
  también mayor retardo de fase y posibles artefactos (ringing) en la respuesta
  al escalón. Para audio, órdenes 4–6 son el compromiso habitual.
- **Orden de aplicación:** es indiferente matemáticamente (paso alto + paso bajo
  es equivalente a un único paso banda), pero aplicarlos por separado permite
  inspeccionar el resultado intermedio, como hemos hecho aquí.
